In [ ]:
# Step 1: Create and setup Hugging Face API Token

In [1]:
# Step 2: Install Required Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl=0.14.0 peft==0.14.0 xformers==0.0.28.post3
!pip install torch=2.5.1 torchvision =0.20.1 -index-url https://download.pytorch.org/whl/cu124

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-6lm223ld/unsloth_e973c956e4d144629ed71ee2b4c89dc8
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-6lm223ld/unsloth_e973c956e4d144629ed71ee2b4c89dc8
  Resolved https://github.com/unslothai/unsloth.git to commit 0096445910418fe051d4b3eb0f866ee781344b76
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.3/124.3 kB 11.7 MB/s eta 0:00:

ERROR: Invalid requirement: 'trl=0.14.0': Expected end or semicolon (after name and no valid version specifier)
    trl=0.14.0
       ^
Hint: = is not a valid operator. Did you mean == ?
ERROR: Invalid requirement: 'torch=2.5.1': Expected end or semicolon (after name and no valid version specifier)
    torch=2.5.1
         ^
Hint: = is not a valid operator. Did you mean == ?


In [1]:
# Step 3: Import Necessary Libraries
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from huggingface_hub import login
from transformers import TrainingArguments
from datasets import load_dataset
import wandb

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
from google.colab import userdata
ha_token = userdata.get('HA_TOKEN')
login(ha_token)

In [3]:
# Check GPU Availability
import torch
print("CUDA Available: ", torch.cuda.is_available())
print("GPU Device: ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA Available:  True
GPU Device:  Tesla T4


In [4]:
# Step 5: Setup pretrained DeepSeek-R1
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
max_sequence_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_sequence_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token = ha_token
)

==((====))==  Unsloth 2025.4.4: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [5]:
# Step 6:
prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addressess the request.
Before crafting the answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Task:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Query:
{}


### Answer:
<think>{}
"""

In [34]:
# Step 7: Run inference on the model

# Define a test question
question = """
A 61-year-old women with a long history of involuntary urine loss during activities like coughing or
sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
what would cystometry most likely reveal about her residual volume and detrusor contractions?
"""

FastLanguageModel.for_inference(model)

# Tokenize the input
inputs= tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200,
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.batch_decode(outputs)
print(response)

IndexError: Replacement index 2 out of range for positional args tuple

In [7]:
print(response[0])

<｜begin▁of▁sentence｜>
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addressess the request.
Before crafting the answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Task:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Query:

A 61-year-old women with a long history of involuntary urine loss during activities like coughing or
sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
what would cystometry most likely reveal about her residual volume and detrusor contractions?



### Answer:
<think>
Okay, so I have a question about a 61-year-old woman who has a history of involuntary urine loss, specifically d

In [8]:
print(response[0].split("### Answer:")[1])


<think>
Okay, so I have a question about a 61-year-old woman who has a history of involuntary urine loss, specifically during activities like coughing or sneezing, but she doesn't leak at night. She undergoes a gynecological exam and a Q-tip test. I need to figure out what a cystometry would reveal about her residual volume and detrusor contractions. 

First, I should probably start by understanding the terms. I know that cystometry is a test to assess bladder function. It involves inserting a catheter to measure how much urine is left in the bladder (residual volume) and how the bladder contracts when it's filled (detrusor contractions). 

The patient has involuntary urine loss during activities like coughing or sneezing. That sounds like she might have urgency or incontinence during those activities. But she doesn't leak at night, which suggests she doesn't have nighttime urinary incontinence, possibly indicating her bladder is continent at night. 

Now, considering she underwent a 

In [9]:
# Step 8: Setup fine-tuning

# Load Dataset
medical_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train[:500]", trust_remote_code=True)

README.md:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

In [10]:
medical_dataset[0]

{'Question': 'Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?',
 'Complex_CoT': "Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot from the leg somehow travels to the l

In [12]:
EOS_TOKEN =tokenizer.eos_token #Define EOS_TOKEN which tells the model when to stop generating text during training
EOS_TOKEN

'<｜end▁of▁sentence｜>'

In [14]:
# Update the prompt_Style for training
prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addressess the request.
Before crafting the answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}


### Response:
<think>
{}
<think>
{}
"""

In [15]:
#Prepare the data for fine-tuning

def preprocess_function(examples):
  inputs = examples["Question"]
  cots = examples["Complex_CoT"]
  outputs = examples["Response"]

  texts = []
  for input, cot, output in zip(inputs, cots, outputs):
    text = prompt_style.format(input, cot, output) + EOS_TOKEN
    texts.append(text)

  return {
      "texts": texts,
  }

In [16]:
finetune_dataset = medical_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [17]:
finetune_dataset["texts"][0]

"\nBelow is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addressess the request.\nBefore crafting the answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.\n\n### Instruction:\nYou are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.\n\n### Question:\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?\n\n\n### Response:\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, 

In [18]:
# Step 9: Setup/Apply LoRA finetuning to the model

model_lora = FastLanguageModel.get_peft_model(
    model = model,
    r = 16, #LoRA Number of adapters to train, depend on case may be more or less
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3047,
    use_rslora = False,
    loftq_config = None
)

Unsloth 2025.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [21]:
# Add this before creating the trainer
if hasattr(model, '_unwrapped_old_generate'):
  del model._unwrapped_old_generate

In [22]:
trainer = SFTTrainer(
    model = model_lora,
    tokenizer = tokenizer,
    train_dataset = finetune_dataset,
    dataset_text_field = "texts",
    max_seq_length = max_sequence_length,
    dataset_num_proc = 1, # No of GPU Available)

    # Define training args
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),

)

Unsloth: Tokenizing ["texts"]:   0%|          | 0/500 [00:00<?, ? examples/s]

In [26]:
# Setup WANDB
from google.colab import userdata
wandb_key = userdata.get('WANDB_API_TOKEN')
wandb.login(key=wandb_key)
run = wandb.init(
    project = 'Fine-tune-DeepSeek-R1-on-Medical-CoT-Dataset',
    job_type = "training",
      anonymous ="allow"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [27]:
# Start the fine-tuning process
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.365100
20,0.012200
30,0.001200
40,0.001900
50,0.000200
60,-0.000100


In [28]:
wandb.finish()

train/epoch,▁▂▄▅▇██
train/global_step,▁▂▄▅▇██
train/grad_norm,█▂▂▁▂▁
train/learning_rate,█▇▅▄▂▁
train/loss,█▁▁▁▁▁
total_flos,1.7087225436389376e+16
train/epoch,0.96
train/global_step,60
train/grad_norm,0.00397
train/learning_rate,1e-05
train/loss,-0.0001


In [35]:

# Step10: Testing after fine-tuning
Question = """A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing
              but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
              what would cystometry most likely reveal about her residual volume and detrusor contractions?"""

FastLanguageModel.for_inference(model_lora)

# Tokenize the input
inputs = tokenizer([prompt_style.format(Question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model_lora.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200,
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.batch_decode(outputs)

print(response)

IndexError: Replacement index 2 out of range for positional args tuple

In [38]:
# Step10: Testing after fine-tuning
Question = """A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing
              but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
              what would cystometry most likely reveal about her residual volume and detrusor contractions?"""

FastLanguageModel.for_inference(model_lora)

# Use the original prompt_style for inference:
prompt_style_inference = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addressess the request.
Before crafting the answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Task:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Query:
{}


### Answer:
<think>{}
"""


# Tokenize the input using the inference prompt_style
inputs = tokenizer([prompt_style_inference.format(Question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model_lora.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200,
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.batch_decode(outputs)

print(response)

['<｜begin▁of▁sentence｜>\nBelow is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addressess the request.\nBefore crafting the answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.\n\n### Task:\nYou are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.\n\n### Query:\nA 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing\n              but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,\n              what would cystometry most likely reveal about her residual volume and detrusor contractions?\n\n\n### Answer:\n<think>\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\

In [39]:
print(response[0].split("### Answer:")[1])


<think>





















































.

.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.

.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.




.




.




.




.




.




.




.



























































In [41]:

question = """A 59-year-old man presents with a fever, chills, night sweats, and generalized fatigue,
              and is found to have a 12 mm vegetation on the aortic valve. Blood cultures indicate gram-positive, catalase-negative,
              gamma-hemolytic cocci in chains that do not grow in a 6.5% NaCl medium.
              What is the most likely predisposing factor for this patient's condition?"""

FastLanguageModel.for_inference(model_lora)

# Use the original prompt_style for inference:
prompt_style_inference = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addressess the request.
Before crafting the answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Task:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Query:
{}


### Answer:
<think>{}
"""


# Tokenize the input using the inference prompt_style
inputs = tokenizer([prompt_style_inference.format(Question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model_lora.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200,
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.batch_decode(outputs)


print(response[0].split("### Answer:")[1])


<think>




















































.

.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.
.

.

.

.

.

.

.

.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.


.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.



.




.




.




.




.




.




.




.





.




























































